In [23]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')



In [24]:
pip install -U langchain langchain-community sentence-transformers

In [25]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
# load data
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/cricket_stats.csv")
documents = loader.load()


In [28]:
pip install -U langchain-text-splitters

In [29]:
 pip install -U langchain-text-splitters

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0
)

documents = text_splitter.split_documents(documents)



In [31]:
pip install faiss-cpu

In [32]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(documents, embeddings)



In [33]:
retriever = vectorstore.as_retriever()


In [34]:
pip install langchain-groq

In [35]:
# =========================
# 1. IMPORTS (LATEST)
# =========================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq


# =========================
# 2. LLM (GROQ)
# =========================
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",   # ✅ NEW working model
    temperature=0
)

# =========================
# 3. FORMAT FUNCTION
# =========================
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# =========================
# 4. PROMPT
# =========================
template = """
You are a helpful assistant that answers questions based on the provided context.
Use the provided context to answer the question.

Question: {input}
Context: {context}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)


# =========================
# 5. RETRIEVER (IMPROVED)
# =========================
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


# =========================
# 6. RAG PIPELINE
# =========================
rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


# =========================
# 7. TEST QUERY
# =========================


In [40]:
from IPython.display import display, Markdown
# response
response = rag_chain.invoke("can u give any 3 indian cricketers")
display(Markdown(response))


Based on the provided context, here are 3 Indian cricketers:

1. Jasprit Bumrah
2. (However, there is no information about other Indian cricketers in the provided context. But we can see that Jasprit Bumrah is an Indian cricketer. To provide 3 Indian cricketers, we need more information about other Indian cricketers. Unfortunately, the provided context does not contain any information about other Indian cricketers.)

However, if we consider the context as a database of cricketers and we need to find 3 Indian cricketers, we can assume that the context is incomplete and we need to find Indian cricketers from other sources. 

But if we consider the context as a database of cricketers and we need to find 3 Indian cricketers from the provided context, we can say that there is only 1 Indian cricketer in the provided context, which is Jasprit Bumrah.